In [72]:
import pandas as pd
import re

In [73]:
# logic.py
"""
Python port of your loan suggestion logic from logic.js.
Functions:
  - update_with_la(records, la)
  - update_with_da(records, da)
  - query_complex(scenarios, deposit_amount=None, repayment_duration=None,
                  deposit_duration=None, interest_rate=None, credit_score=None)
  - calculate_sort_order(loan)

Expect each record to be a dict with fields matching your JS schema.
Requires a JSON file 'parameters_weights.json' in the same directory.
"""
import json
from typing import List, Dict, Any, Optional

# Load parameter weights once
def _load_weights() -> Dict[str, Any]:
    with open('parameters_weights.json', 'r', encoding='utf-8') as f:
        return json.load(f)

_parameters_weights = _load_weights()


In [74]:
import json
from typing import List, Dict, Any, Optional

# Load parameter weights once
def _load_record() -> Dict[str, Any]:
    with open('MEC-LoanRecomn_Scenarios-V15.json', 'r', encoding='utf-8') as f:
        return json.load(f)

_records = _load_record()


In [75]:
_records

[{'id': 1,
  'nickname': 'بهان(بدون ضامن)',
  'package_name': 'فرابانک',
  'contract_type': 'مرابحه',
  'granted_method': 'واریز به حساب',
  'loan_amount_limit': 1000000000,
  'deposit_duration': 3,
  'interest_rate': 23,
  'repayment_duration': 60,
  'loan_coefficient': 1000,
  'credit_score': 'A',
  'minimum_deposit_amount': 'nan',
  'maximum_deposit_amount': 'nan',
  'minimum_loan_amount': 100000000,
  'guarantee': 'خیر',
  'receiving_channel': 'غیر حضوری'},
 {'id': 2,
  'nickname': 'بهان(بدون ضامن)',
  'package_name': 'فرابانک',
  'contract_type': 'مرابحه',
  'granted_method': 'واریز به حساب',
  'loan_amount_limit': 1000000000,
  'deposit_duration': 3,
  'interest_rate': 23,
  'repayment_duration': 60,
  'loan_coefficient': 1000,
  'credit_score': 'B',
  'minimum_deposit_amount': 'nan',
  'maximum_deposit_amount': 'nan',
  'minimum_loan_amount': 100000000,
  'guarantee': 'خیر',
  'receiving_channel': 'غیر حضوری'},
 {'id': 3,
  'nickname': 'شایان',
  'package_name': 'شایان یک',
  'c

In [76]:
def calculate_sort_order(loan: Dict[str, Any]) -> None:
    """
    Mutates loan by adding a 'sortOrder' key based on weighted criteria,
    faithfully mirroring the JS logic.
    """
    # Determine bucket
    loanAmountKey = 'out_of_range'
    la = loan.get('loan_amount', 0)
    if la <= 500_000_000:
        loanAmountKey = '1-50'
    elif la <= 1_000_000_000:
        loanAmountKey = '50-100'
    elif la <= 1_500_000_000:
        loanAmountKey = '100-150'
    elif la <= 2_000_000_000:
        loanAmountKey = '150-200'
    elif la <= 2_500_000_000:
        loanAmountKey = '200-250'
    elif la <= 3_000_000_000:
        loanAmountKey = '250-300'

    pw = _parameters_weights
    # Extract individual weights
    ir_key = str(loan.get('interest_rate', ''))
    rd_key = str(loan.get('repayment_duration', ''))
    dd_key = str(loan.get('deposit_duration', ''))
    # CS: first A-E or N
    cs_match = re.search(r'[ABCDE]', loan.get('credit_score', '') or '')
    cs_key = cs_match.group(0) if cs_match else 'N'

    ir_value = pw['IR'][loanAmountKey].get(ir_key, 0)
    rd_value = pw['RD'][loanAmountKey].get(rd_key, 0)
    w_type_coef = pw['w_type'][loanAmountKey].get(loan.get('nickname', ''), 1)
    dd_value = pw['DD'].get(dd_key, 0)
    cs_value = pw['CS'].get(cs_key, 0)

    # Global weights
    w = pw['w']
    coef = (
        ir_value * w['IR_score'] +
        rd_value * w['RD_score'] +
        dd_value * w['DD_score'] +
        cs_value * w['CS_score']
    )
    # Business weight
    w_business = pw['w_business'].get(loan.get('nickname', ''), 1)

    # Final score
    # loan['w_IR_score'] = w['IR_score']
    # loan['ir_value'] = ir_value
    # loan['w_RD_score'] = w['RD_score']
    # loan['rd_value'] = rd_value
    # loan['w_DD_score'] = w['DD_score']
    # loan['dd_value'] = dd_value
    # loan['w_CS_score'] = w['CS_score']
    # loan['cs_value'] = cs_value
    
    # loan['w_type_coef'] = w_type_coef
    # loan['coef'] = coef
    # loan['w_business'] = w_business
    loan['sortOrder'] = coef * w_type_coef * w_business


In [77]:

def update_with_la(records: List[Dict[str, Any]], la: float) -> List[Dict[str, Any]]:
    """
    Given desired loan amount 'la', update each record's
    loan_amount, monthly_repayment, deposit_amount,
    then filter and return valid records.
    """
    for rec in records:
        # calculate monthly repayment
        r = rec.get('repayment_duration', 0)
        ir_monthly = rec.get('interest_rate', 0) / (12 * 100)
        num = la * ir_monthly * (1 + ir_monthly) ** r
        den = (1 + ir_monthly) ** r - 1
        rec['loan_amount'] = la
        rec['monthly_repayment'] = num / den   # change to Rial
        rec['deposit_amount'] = (la / rec.get('loan_coefficient', 1)) * 100
        calculate_sort_order(rec)

    def valid(rec: Dict[str, Any]) -> bool:
        max_dep = rec.get('maximum_deposit_amount')
        if max_dep and max_dep.lower() != 'nan':
            try:
                if rec.get('deposit_amount') > int(max_dep):
                    return False
            except ValueError:
                pass
        la_lim = rec.get('loan_amount_limit', float('inf'))
        min_la = rec.get('minimum_loan_amount', 0)
        return min_la <= rec.get('loan_amount') <= la_lim
    valid_records = [r for r in records if valid(r)]
    # return valid_records
    return sorted(valid_records, key=lambda x: x.get('sortOrder', 0), reverse=True)

In [78]:
# df_all = pd.DataFrame(_records)
# df_all.head()

In [79]:
# df = pd.DataFrame(update_with_la(_records, la = 700_000_000))
# df.head(15)

In [80]:
# len(update_with_la(_records, la = 700_000_000))

In [81]:
def update_with_da(records: List[Dict[str, Any]], da: float) -> List[Dict[str, Any]]:
    """
    Given deposit amount 'da', update each record's
    loan_amount, monthly_repayment, deposit_amount,
    then filter and return valid records.
    """
    for rec in records:
        coeff = rec.get('loan_coefficient', 0) / 100
        la = coeff * da
        r = rec.get('repayment_duration', 0)
        ir_monthly = rec.get('interest_rate', 0) / (12 * 100)
        num = la * ir_monthly * (1 + ir_monthly) ** r
        den = (1 + ir_monthly) ** r - 1
        rec['loan_amount'] = la
        rec['monthly_repayment'] = num / den  # change to Rial
        rec['deposit_amount'] = da
        calculate_sort_order(rec)

    def valid(rec: Dict[str, Any]) -> bool:
        max_dep = rec.get('maximum_deposit_amount')
        if max_dep and max_dep.lower() != 'nan':
            try:
                if rec.get('deposit_amount') > int(max_dep):
                    return False
            except ValueError:
                pass
        la_lim = rec.get('loan_amount_limit', float('inf'))
        min_la = rec.get('minimum_loan_amount', 0)
        return min_la <= rec.get('loan_amount') <= la_lim

    valid_records = [r for r in records if valid(r)]
    # Sort descending by sortOrder
    return sorted(valid_records, key=lambda x: x.get('sortOrder', 0), reverse=True)



In [82]:
# print(update_with_da(_records, da = 200_000_000))

In [83]:
# len(update_with_da(_records, da = 200_000_000))

In [84]:
def query_complex(
    scenarios: List[Dict[str, Any]],
    deposit_amount: Optional[float] = None,
    repayment_duration: Optional[int] = None,
    deposit_duration: Optional[int] = None,
    interest_rate: Optional[float] = None,
    credit_score: Optional[str] = None
) -> List[Dict[str, Any]]:
    """
    Filter scenarios based on provided parameters.
    """
    matches = []
    for rec in scenarios:
        # deposit range: up to 1.6x
        cond_da = (deposit_amount is None or
                   rec.get('deposit_amount', 0) <= deposit_amount * 1.6)
        cond_rd = (repayment_duration is None or
                   rec.get('repayment_duration') == repayment_duration)
        cond_dep = (deposit_duration is None or
                    rec.get('deposit_duration') == deposit_duration)
        cond_ir = (interest_rate is None or
                   rec.get('interest_rate') == interest_rate)
        cs_field = rec.get('credit_score', '')
        cond_cs = (credit_score is None or
                   (credit_score in cs_field) or
                   (credit_score == 'N' and 'فاقد رتبه' in cs_field))
        if cond_da and cond_rd and cond_dep and cond_ir and cond_cs:
            matches.append(rec)
    return sorted(matches, key=lambda x: x.get('sortOrder', 0), reverse=True)




In [85]:

def get_query_params( _records, 
    deposit__amount: Optional[float] = None,
    repayment__duration: Optional[int] = None,
    deposit__duration: Optional[int] = None,
    interest__rate: Optional[float] = None,
    credit__score: Optional[str] = None,
    loan__amount: Optional[float] = None
) -> List[Dict[str, Any]]:   #Input values are Toman


    if loan__amount:
        scenarios = update_with_la(_records, loan__amount)
    elif deposit__amount and loan__amount is None:
        scenarios = update_with_da(_records, deposit__amount)
    else:
        scenarios = _records

    report = query_complex(
        scenarios,
        deposit_amount=deposit__amount,
        repayment_duration=repayment__duration,
        deposit_duration=deposit__duration,
        interest_rate=interest__rate,
        credit_score=credit__score
    )

    return report

In [86]:
result = get_query_params( _records, 
    loan__amount = 200_000_000,
    deposit__amount= None,
    repayment__duration = None,
    deposit__duration = None,
    interest__rate = 18,
    credit__score = None,
    )

In [87]:
len(result)

30

In [88]:
nickname_0 = result[0].get('nickname', 'نامشخص')
loan_amount_0 = int(result[0].get('loan_amount', 0) / 10_000_000)
monthly_repayment_0 = int(result[0].get('monthly_repayment', 0) // 10)
deposit_amount_0=int(result[0].get('deposit_amount', 0) / 10_000_000)
interest_rate_0=result[0].get('interest_rate', 0)
repayment_duration_0=result[0].get('repayment_duration', 0)
deposit_duration_0=result[0].get('deposit_duration', 0)
credit_score_0=result[0].get('credit_score', 'نامشخص')

In [89]:
result[0]

{'id': 42,
 'nickname': 'شایان',
 'package_name': 'شایان یک',
 'contract_type': 'مرابحه/جعاله',
 'granted_method': 'واریز به حساب',
 'loan_amount_limit': 3000000000,
 'deposit_duration': 3,
 'interest_rate': 18,
 'repayment_duration': 12,
 'loan_coefficient': 80,
 'credit_score': 'B',
 'minimum_deposit_amount': 'nan',
 'maximum_deposit_amount': '15000000000',
 'minimum_loan_amount': 100000000,
 'guarantee': 'بله',
 'receiving_channel': 'غیر حضوری/ حضوری',
 'loan_amount': 200000000,
 'monthly_repayment': 18335998.58124589,
 'deposit_amount': 250000000.0,
 'sortOrder': 0.044667}

In [90]:
msg_filter = f"""{loan_amount_0} میلیون تومان وام {nickname_0} با کارمزد {interest_rate_0} درصد،
 نیاز به {deposit_amount_0}  میلیون تومان سپرده {deposit_duration_0} ماهه،
 با اقساط {monthly_repayment_0}  تومان در {repayment_duration_0} ماه
 با رتبه اعتباری {credit_score_0}.
 """


In [91]:
print(msg_filter)

20 میلیون تومان وام شایان با کارمزد 18 درصد،
 نیاز به 25  میلیون تومان سپرده 3 ماهه،
 با اقساط 1833599  تومان در 12 ماه
 با رتبه اعتباری B.
 


In [92]:
11_612_920

11612920

In [93]:
pd.DataFrame(result).head(10)

,id,nickname,package_name,contract_type,granted_method,loan_amount_limit,deposit_duration,interest_rate,repayment_duration,loan_coefficient,credit_score,minimum_deposit_amount,maximum_deposit_amount,minimum_loan_amount,guarantee,receiving_channel,loan_amount,monthly_repayment,deposit_amount,sortOrder
0,42,شایان,شایان یک,مرابحه/جعاله,واریز به حساب,3000000000,3,18,12,80,B,nan,15000000000,100000000,بله,غیر حضوری/ حضوری,200000000,1.833600e+07,2.500000e+08,0.044667
1,43,شایان,شایان یک,مرابحه,کارت اعتباری,3000000000,3,18,12,100,B,nan,15000000000,100000000,بله,غیر حضوری/ حضوری,200000000,1.833600e+07,2.000000e+08,0.044667
2,10,شایان,شایان یک,مرابحه/جعاله,واریز به حساب,3000000000,3,18,12,80,A,nan,15000000000,100000000,بله,غیر حضوری/ حضوری,200000000,1.833600e+07,2.500000e+08,0.041517
3,11,شایان,شایان یک,مرابحه,کارت اعتباری,3000000000,3,18,12,100,A,nan,15000000000,100000000,بله,غیر حضوری/ حضوری,200000000,1.833600e+07,2.000000e+08,0.041517
4,54,شایان,شایان یک,مرابحه/جعاله,واریز به حساب,3000000000,6,18,12,170,B,nan,15000000000,100000000,بله,غیر حضوری/ حضوری,200000000,1.833600e+07,1.176471e+08,0.040467
5,55,شایان,شایان یک,مرابحه,کارت اعتباری,3000000000,6,18,12,200,B,nan,15000000000,100000000,بله,غیر حضوری/ حضوری,200000000,1.833600e+07,1.000000e+08,0.040467
6,57,شایان,شایان یک,مرابحه/جعاله,واریز به حساب,3000000000,6,18,24,75,B,nan,15000000000,100000000,بله,غیر حضوری/ حضوری,200000000,9.984820e+06,2.666667e+08,0.040467
7,58,شایان,شایان یک,مرابحه,کارت اعتباری,3000000000,6,18,24,70,B,nan,15000000000,100000000,بله,غیر حضوری/ حضوری,200000000,9.984820e+06,2.857143e+08,0.040467
8,71,شایان,شایان یک,مرابحه/جعاله,واریز به حساب,3000000000,3,18,12,70,C,nan,15000000000,100000000,بله,غیر حضوری/ حضوری,200000000,1.833600e+07,2.857143e+08,0.039942
9,72,شایان,شایان یک,مرابحه,کارت اعتباری,3000000000,3,18,12,90,C,nan,15000000000,100000000,بله,غیر حضوری/ حضوری,200000000,1.833600e+07,2.222222e+08,0.039942


# Final version (Deployed V)

In [94]:
import pandas as pd
#------------------------------------
# pd.options.display.max_columns= None
# pd.options.display.max_colwidth= 2
pd.options.display.max_rows = None

#------------------------------------

In [95]:
import pandas as pd
import re
import json
from typing import List, Dict, Any, Optional

# Load parameter weights once
def _load_weights() -> Dict[str, Any]:
    with open('parameters_weights.json', 'r', encoding='utf-8') as f:
        return json.load(f)

_parameters_weights = _load_weights()



def calculate_sort_order(loan: Dict[str, Any]) -> None:
    
    coef_table = loan.get('loan_coefficient',0)
    
    # Determine bucket
    loanAmountKey = 'out_of_range'
    la = loan.get('loan_amount', 0)
    if la <= 500_000_000:
        loanAmountKey = '1-50'
    elif la <= 1_000_000_000:
        loanAmountKey = '50-100'
    elif la <= 1_500_000_000:
        loanAmountKey = '100-150'
    elif la <= 2_000_000_000:
        loanAmountKey = '150-200'
    elif la <= 2_500_000_000:
        loanAmountKey = '200-250'
    elif la <= 3_000_000_000:
        loanAmountKey = '250-300'

    pw = _parameters_weights
    # Extract individual weights
    ir_key = str(loan.get('interest_rate', ''))
    rd_key = str(loan.get('repayment_duration', ''))
    dd_key = str(loan.get('deposit_duration', ''))
    # CS: first A-E or N
    cs_match = re.search(r'[ABCDE]', loan.get('credit_score', '') or '')
    cs_key = cs_match.group(0) if cs_match else 'N'

    ir_value = pw['IR'][loanAmountKey].get(ir_key, 0)
    rd_value = pw['RD'][loanAmountKey].get(rd_key, 0)
    w_type_coef = pw['w_type'][loanAmountKey].get(loan.get('nickname', ''), 1)
    dd_value = pw['DD'].get(dd_key, 0)
    cs_value = pw['CS'].get(cs_key, 0)

    # Global weights
    w = pw['w']
    coef = (
        ir_value * w['IR_score'] +
        rd_value * w['RD_score'] +
        dd_value * w['DD_score'] +
        cs_value * w['CS_score']
    )
    # Business weight
    w_business = pw['w_business'].get(loan.get('nickname', ''), 1)
    # Calculate final score
    # loan['sortOrder'] = coef * w_type_coef * w_business
    loan['sortOrder'] = coef * w_type_coef * coef_table
    loan['pre_sortOrder'] = coef * w_type_coef 





def update_with_la(records: List[Dict[str, Any]], la: float) -> List[Dict[str, Any]]:
    """
    Given desired loan amount 'la', update each record's
    loan_amount, monthly_repayment, deposit_amount,
    then filter and return valid records.
    """
    for rec in records:
        # calculate monthly repayment
        r = rec.get('repayment_duration', 0)
        ir_monthly = rec.get('interest_rate', 0) / (12 * 100)
        num = la * ir_monthly * (1 + ir_monthly) ** r
        den = (1 + ir_monthly) ** r - 1
        rec['loan_amount'] = la
        rec['monthly_repayment'] = num / den   # change to Rial
        rec['deposit_amount'] = (la / rec.get('loan_coefficient', 1)) * 100
        calculate_sort_order(rec)

    def valid(rec: Dict[str, Any]) -> bool:
        max_dep = rec.get('maximum_deposit_amount')
        if max_dep and max_dep.lower() != 'nan':
            try:
                if rec.get('deposit_amount') > int(max_dep):
                    return False
            except ValueError:
                pass
        la_lim = rec.get('loan_amount_limit', float('inf'))
        min_la = rec.get('minimum_loan_amount', 0)
        return min_la <= rec.get('loan_amount') <= la_lim
    valid_records = [r for r in records if valid(r)]
    # return valid_records
    return sorted(valid_records, key=lambda x: x.get('sortOrder', 0), reverse=True)


def update_with_da(records: List[Dict[str, Any]], da: float) -> List[Dict[str, Any]]:
    """
    Given deposit amount 'da', update each record's
    loan_amount, monthly_repayment, deposit_amount,
    then filter and return valid records.
    """
    for rec in records:
        coeff = rec.get('loan_coefficient', 0) / 100
        la = coeff * da
        r = rec.get('repayment_duration', 0)
        ir_monthly = rec.get('interest_rate', 0) / (12 * 100)
        num = la * ir_monthly * (1 + ir_monthly) ** r
        den = (1 + ir_monthly) ** r - 1
        rec['loan_amount'] = la
        rec['monthly_repayment'] = num / den  # change to Rial
        rec['deposit_amount'] = da
        calculate_sort_order(rec)

    def valid(rec: Dict[str, Any]) -> bool:
        max_dep = rec.get('maximum_deposit_amount')
        if max_dep and max_dep.lower() != 'nan':
            try:
                if rec.get('deposit_amount') > int(max_dep):
                    return False
            except ValueError:
                pass
        la_lim = rec.get('loan_amount_limit', float('inf'))
        min_la = rec.get('minimum_loan_amount', 0)
        return min_la <= rec.get('loan_amount') <= la_lim

    valid_records = [r for r in records if valid(r)]
    # Sort descending by sortOrder
    return sorted(valid_records, key=lambda x: x.get('sortOrder', 0), reverse=True)



def query_complex(
    scenarios: List[Dict[str, Any]],
    deposit_amount: Optional[float] = None,
    repayment_duration: Optional[int] = None,
    deposit_duration: Optional[int] = None,
    interest_rate: Optional[float] = None,
    credit_score: Optional[str] = None
) -> List[Dict[str, Any]]:
    """
    Filter scenarios based on provided parameters.
    """
    matches = []
    for rec in scenarios:
        # deposit range: up to 1.6x
        cond_da = (deposit_amount is None or
                   rec.get('deposit_amount', 0) <= deposit_amount * 1.6)
        cond_rd = (repayment_duration is None or
                   rec.get('repayment_duration') == repayment_duration)
        cond_dep = (deposit_duration is None or
                    rec.get('deposit_duration') == deposit_duration)
        cond_ir = (interest_rate is None or
                   rec.get('interest_rate') == interest_rate)
        cs_field = rec.get('credit_score', '')
        cond_cs = (credit_score is None or
                   (credit_score in cs_field) or
                   (credit_score == 'N' and 'فاقد رتبه' in cs_field))
        if cond_da and cond_rd and cond_dep and cond_ir and cond_cs:
            matches.append(rec)
    return sorted(matches, key=lambda x: x.get('sortOrder', 0), reverse=True)



In [96]:
import json
from typing import List, Dict, Any, Optional

# Load parameter weights once
def load_record() -> Dict[str, Any]:
    with open('MEC-LoanRecomn_Scenarios-V16.json', 'r', encoding='utf-8') as f:
        return json.load(f)

_records = load_record()



def get_query_params( _records, 
    deposit__amount: Optional[float] = None,
    repayment__duration: Optional[int] = None,
    deposit__duration: Optional[int] = None,
    interest__rate: Optional[float] = None,
    credit__score: Optional[str] = None,
    loan__amount: Optional[float] = None
) -> List[Dict[str, Any]]:   #Input values are Toman


    if loan__amount:
        scenarios = update_with_la(_records, loan__amount)
    elif deposit__amount and loan__amount is None:
        scenarios = update_with_da(_records, deposit__amount)
    else:
        scenarios = _records

    report = query_complex(
        scenarios,
        deposit_amount=deposit__amount,
        repayment_duration=repayment__duration,
        deposit_duration=deposit__duration,
        interest_rate=interest__rate,
        credit_score=credit__score
    )

    

    # msg= f"تعداد {len(report)} پیشنهاد وام برای شما پیدا شد.\n\n"
    loan_number = len(report)
    return report, loan_number


In [97]:
result , number = get_query_params( _records, 
    loan__amount = 1_000_000_000,
    deposit__amount= None,
    repayment__duration = None,
    deposit__duration = None,
    interest__rate = None,
    credit__score = None,
    )

In [98]:
print(number)

159


In [99]:
df_new = pd.DataFrame(result)

In [100]:
df_reduced = df_new.drop(["granted_method","loan_amount_limit","maximum_deposit_amount","minimum_deposit_amount","minimum_loan_amount","guarantee","receiving_channel","contract_type","package_name"], axis = 1)

In [101]:
df_reduced

,id,nickname,deposit_duration,interest_rate,repayment_duration,loan_coefficient,credit_score,loan_amount,monthly_repayment,deposit_amount,sortOrder,pre_sortOrder
0,2,بهان(بدون ضامن),3,23,60,1000,B,1000000000,2.819047e+07,1.000000e+08,101.67000,0.101670
1,1,بهان(بدون ضامن),3,23,60,1000,A,1000000000,2.819047e+07,1.000000e+08,97.17000,0.097170
2,101,فرابانک,3,23,36,1000,B,1000000000,3.870972e+07,1.000000e+08,61.36200,0.061362
3,100,فرابانک,3,23,36,1000,A,1000000000,3.870972e+07,1.000000e+08,58.66200,0.058662
4,102,فرابانک,3,23,36,1000,C,1000000000,3.870972e+07,1.000000e+08,57.31200,0.057312
5,45,شایان,3,23,12,450,B,1000000000,9.407632e+07,2.222222e+08,26.96490,0.059922
6,13,شایان,3,23,12,450,A,1000000000,9.407632e+07,2.222222e+08,25.74990,0.057222
7,137,نیک‌وام,12,4,12,320,B,1000000000,8.514990e+07,3.125000e+08,22.84800,0.071400
8,118,نیک‌وام,12,4,12,320,A,1000000000,8.514990e+07,3.125000e+08,21.21600,0.066300
9,156,نیک‌وام,12,4,12,320,C,1000000000,8.514990e+07,3.125000e+08,20.40000,0.063750


In [102]:
df_reduced ["coef_sort"]=df_reduced["loan_coefficient"]*df_reduced["pre_sortOrder"]

In [103]:
df_reduced.head(100)

,id,nickname,deposit_duration,interest_rate,repayment_duration,loan_coefficient,credit_score,loan_amount,monthly_repayment,deposit_amount,sortOrder,pre_sortOrder,coef_sort
0,2,بهان(بدون ضامن),3,23,60,1000,B,1000000000,2.819047e+07,1.000000e+08,101.67000,0.101670,101.67000
1,1,بهان(بدون ضامن),3,23,60,1000,A,1000000000,2.819047e+07,1.000000e+08,97.17000,0.097170,97.17000
2,101,فرابانک,3,23,36,1000,B,1000000000,3.870972e+07,1.000000e+08,61.36200,0.061362,61.36200
3,100,فرابانک,3,23,36,1000,A,1000000000,3.870972e+07,1.000000e+08,58.66200,0.058662,58.66200
4,102,فرابانک,3,23,36,1000,C,1000000000,3.870972e+07,1.000000e+08,57.31200,0.057312,57.31200
5,45,شایان,3,23,12,450,B,1000000000,9.407632e+07,2.222222e+08,26.96490,0.059922,26.96490
6,13,شایان,3,23,12,450,A,1000000000,9.407632e+07,2.222222e+08,25.74990,0.057222,25.74990
7,137,نیک‌وام,12,4,12,320,B,1000000000,8.514990e+07,3.125000e+08,22.84800,0.071400,22.84800
8,118,نیک‌وام,12,4,12,320,A,1000000000,8.514990e+07,3.125000e+08,21.21600,0.066300,21.21600
9,156,نیک‌وام,12,4,12,320,C,1000000000,8.514990e+07,3.125000e+08,20.40000,0.063750,20.40000


In [104]:
df_reduced.sort_values(by="coef_sort", inplace=True, ascending=False)

In [105]:
df_reduced

,id,nickname,deposit_duration,interest_rate,repayment_duration,loan_coefficient,credit_score,loan_amount,monthly_repayment,deposit_amount,sortOrder,pre_sortOrder,coef_sort
0,2,بهان(بدون ضامن),3,23,60,1000,B,1000000000,2.819047e+07,1.000000e+08,101.67000,0.101670,101.67000
1,1,بهان(بدون ضامن),3,23,60,1000,A,1000000000,2.819047e+07,1.000000e+08,97.17000,0.097170,97.17000
2,101,فرابانک,3,23,36,1000,B,1000000000,3.870972e+07,1.000000e+08,61.36200,0.061362,61.36200
3,100,فرابانک,3,23,36,1000,A,1000000000,3.870972e+07,1.000000e+08,58.66200,0.058662,58.66200
4,102,فرابانک,3,23,36,1000,C,1000000000,3.870972e+07,1.000000e+08,57.31200,0.057312,57.31200
5,45,شایان,3,23,12,450,B,1000000000,9.407632e+07,2.222222e+08,26.96490,0.059922,26.96490
6,13,شایان,3,23,12,450,A,1000000000,9.407632e+07,2.222222e+08,25.74990,0.057222,25.74990
7,137,نیک‌وام,12,4,12,320,B,1000000000,8.514990e+07,3.125000e+08,22.84800,0.071400,22.84800
8,118,نیک‌وام,12,4,12,320,A,1000000000,8.514990e+07,3.125000e+08,21.21600,0.066300,21.21600
9,156,نیک‌وام,12,4,12,320,C,1000000000,8.514990e+07,3.125000e+08,20.40000,0.063750,20.40000


In [108]:
# determining the name of the file
file_name = 'TestSorting.xlsx'

# saving the excel
df_reduced.to_excel(file_name)
print('DataFrame is written to Excel File successfully.')

DataFrame is written to Excel File successfully.


In [106]:
# df_reduced.sort_values(by="loan_coefficient", inplace=True, ascending=False)

In [107]:
# df_reduced